In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os, sys

BASE_PATH = '/content/drive/MyDrive/Colab_Notebooks/DERCo/inputs/raw_fif/osfstorage/EEG-based Reading Experiment/EEG_data/preprocessed'
INPUTS_PATH = os.path.dirname(BASE_PATH)
sys.path.append(INPUTS_PATH)
OUTPUTS_SAVE_PATH = os.path.dirname('/content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data/subjects')
sys.path.append(OUTPUTS_SAVE_PATH)
EXCLUDED = ['QPF42', 'USQ95']
SUBJECTS = sorted([d for d in os.listdir(BASE_PATH) if os.path.isdir(os.path.join(BASE_PATH, d)) and d not in EXCLUDED])


In [3]:
%pip install -U mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 49.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [12]:
import mne
import numpy as np


In [10]:
print(INPUTS_PATH)
print(OUTPUTS_SAVE_PATH)
print(SUBJECTS)

/content/drive/MyDrive/Colab_Notebooks/DERCo/inputs/raw_fif/osfstorage/EEG-based Reading Experiment/EEG_data
/content/drive/MyDrive/Colab_Notebooks/DERCo/processed_data
['ACB71', 'DGR11', 'HMK96', 'JPY86', 'LRK01', 'LYY64', 'MNY88', 'MRB47', 'NXB64', 'OLW10', 'QFT39', 'QPI83', 'RRO98', 'SAB93', 'SIT48', 'TRA37', 'UJM92', 'WAL74', 'WHR08', 'WJX11']


In [11]:
import pandas as pd
import gc

for subj in SUBJECTS:
    subj_X_parts = []
    subj_y_parts = []
    subj_meta_parts = []
    for art in range(5):
        path = f"{INPUTS_PATH}/preprocessed/{subj}/article_{art}/preprocessed_epoch.fif"
        if not os.path.exists(path): 
            print(f"MISSING: {subj} article_{art}")
            continue
        try: 
            ep = mne.read_epochs(path, preload=True, verbose=False)
            ep.apply_baseline(baseline=(-0.2, 0)) # Baseline correction needs the -200 ms to 0 ms window 
            ep.crop(tmin=0.0, tmax=0.8) # Crop to post-stimlus window (0-800 ms)
            meta = ep.metadata

            data = ep.get_data().astype(np.float32)
            del ep
            gc.collect()

            low_mask = (meta['level'] == 'low').values
            high_mask = (meta['level'] == 'high').values

            subj_X_parts.append(data[low_mask])
            subj_y_parts.extend([0] * low_mask.sum())
            
            subj_X_parts.append(data[high_mask])
            subj_y_parts.extend([1] * high_mask.sum())

            subj_meta_parts.append(meta[low_mask])
            subj_meta_parts.append(meta[high_mask])
            
            del data
            gc.collect()

        except Exception as e: 
            print(f"ERROR {subj} article_{art}: {e}")
    if len(subj_X_parts) == 0: 
        print(f"Warning: no data for {subj}, skipping")
        continue
    subj_arr = np.concatenate(subj_X_parts).astype(np.float32)
    subj_y_arr = np.array(subj_y_parts)
    subj_meta = pd.concat(subj_meta_parts, ignore_index=True)

    np.save(f'{OUTPUTS_SAVE_PATH}/subjects/{subj}_X.npy', subj_arr)
    np.save(f'{OUTPUTS_SAVE_PATH}/subjects/{subj}_y.npy', subj_y_arr)
    subj_meta.to_csv(f'{OUTPUTS_SAVE_PATH}/subjects/{subj}_meta.csv', index=False)

    print(f"{subj}: {len(subj_y_arr)} epochs saved to Drive")

    del subj_arr, subj_y_arr, subj_meta, subj_X_parts, subj_y_parts, subj_meta_parts
    gc.collect()
print("\nAll subjects saved.")


/tmp/ipykernel_5099/2355456020.py:14: RuntimeWarning: This filename (/content/drive/MyDrive/Colab_Notebooks/DERCo/inputs/raw_fif/osfstorage/EEG-based Reading Experiment/EEG_data/preprocessed/ACB71/article_0/preprocessed_epoch.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  ep = mne.read_epochs(path, preload=True, verbose=False)


Applying baseline correction (mode: mean)


/tmp/ipykernel_5099/2355456020.py:14: RuntimeWarning: This filename (/content/drive/MyDrive/Colab_Notebooks/DERCo/inputs/raw_fif/osfstorage/EEG-based Reading Experiment/EEG_data/preprocessed/ACB71/article_1/preprocessed_epoch.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  ep = mne.read_epochs(path, preload=True, verbose=False)


Applying baseline correction (mode: mean)


/tmp/ipykernel_5099/2355456020.py:14: RuntimeWarning: This filename (/content/drive/MyDrive/Colab_Notebooks/DERCo/inputs/raw_fif/osfstorage/EEG-based Reading Experiment/EEG_data/preprocessed/ACB71/article_2/preprocessed_epoch.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  ep = mne.read_epochs(path, preload=True, verbose=False)


...

Applying baseline correction (mode: mean)


/tmp/ipykernel_5099/2355456020.py:14: RuntimeWarning: This filename (/content/drive/MyDrive/Colab_Notebooks/DERCo/inputs/raw_fif/osfstorage/EEG-based Reading Experiment/EEG_data/preprocessed/WJX11/article_3/preprocessed_epoch.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  ep = mne.read_epochs(path, preload=True, verbose=False)


Applying baseline correction (mode: mean)


/tmp/ipykernel_5099/2355456020.py:14: RuntimeWarning: This filename (/content/drive/MyDrive/Colab_Notebooks/DERCo/inputs/raw_fif/osfstorage/EEG-based Reading Experiment/EEG_data/preprocessed/WJX11/article_4/preprocessed_epoch.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  ep = mne.read_epochs(path, preload=True, verbose=False)


Applying baseline correction (mode: mean)
WJX11: 1639 epochs saved to Drive

All subjects saved.


In [13]:
SUBJECTS_CHECK = sorted([
    f.replace('_X.npy', '')
    for f in os.listdir(f'{OUTPUTS_SAVE_PATH}/subjects')
    if f.endswith('_X.npy')
])

print(f"Subjects saved: {len(SUBJECTS_CHECK)}")
print(SUBJECTS_CHECK)
print()

for subj in SUBJECTS_CHECK:
    X = np.load(f'{OUTPUTS_SAVE_PATH}/subjects/{subj}_X.npy')
    y = np.load(f'{OUTPUTS_SAVE_PATH}/subjects/{subj}_y.npy')
    meta = pd.read_csv(f'{OUTPUTS_SAVE_PATH}/subjects/{subj}_meta.csv')
    
    assert X.shape[0] == y.shape[0] == len(meta), \
        f"{subj}: X/y/meta length mismatch — X:{X.shape[0]}, y:{y.shape[0]}, meta:{len(meta)}"
    assert X.shape[1:] == (32, 801), \
        f"{subj}: unexpected shape {X.shape}"
    assert set(np.unique(y)) == {0, 1}, \
        f"{subj}: unexpected y values {np.unique(y)}"
    
    print(f"{subj}: {X.shape[0]} epochs | low={( y==0).sum()} high={(y==1).sum()} | shape {X.shape}")

print("\nAll checks passed.")

Subjects saved: 20
['ACB71', 'DGR11', 'HMK96', 'JPY86', 'LRK01', 'LYY64', 'MNY88', 'MRB47', 'NXB64', 'OLW10', 'QFT39', 'QPI83', 'RRO98', 'SAB93', 'SIT48', 'TRA37', 'UJM92', 'WAL74', 'WHR08', 'WJX11']

ACB71: 1560 epochs | low=781 high=779 | shape (1560, 32, 801)
DGR11: 1524 epochs | low=790 high=734 | shape (1524, 32, 801)
HMK96: 1688 epochs | low=852 high=836 | shape (1688, 32, 801)
JPY86: 1565 epochs | low=786 high=779 | shape (1565, 32, 801)
LRK01: 1450 epochs | low=726 high=724 | shape (1450, 32, 801)
LYY64: 1687 epochs | low=853 high=834 | shape (1687, 32, 801)
MNY88: 1588 epochs | low=794 high=794 | shape (1588, 32, 801)
MRB47: 1346 epochs | low=699 high=647 | shape (1346, 32, 801)
NXB64: 1694 epochs | low=870 high=824 | shape (1694, 32, 801)
OLW10: 1631 epochs | low=830 high=801 | shape (1631, 32, 801)
QFT39: 1623 epochs | low=807 high=816 | shape (1623, 32, 801)
QPI83: 1611 epochs | low=839 high=772 | shape (1611, 32, 801)
RRO98: 1740 epochs | low=885 high=855 | shape (1740, 32